## **Upload Sample Dataset**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Load Data**

In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/DS IPYNB/sample.csv")

df = df[["text", "Category"]].dropna()

print(df["Category"].value_counts())

print(df.shape)
df.head()

Category
sports           13430
news             11730
finance           4646
lifestyle         3254
autos             2414
travel            2365
foodanddrink      2323
video             2183
tv                1749
health            1669
weather           1449
music             1119
movies             877
entertainment      653
kids               131
Name: count, dtype: int64
(49992, 2)


,text,Category
0,The real history behind the Stonewall riot: on...,travel
1,Bears 2019 season outlook: Trey Burton 2018 Re...,sports
2,"Renting in Worcester: What will $1,400 get you...",news
3,Wells Fargo (WFC) Stock Sinks As Market Gains:...,finance
4,Last day for Amtrak trains on Indianapolis-to-...,news


In [4]:
from sklearn.preprocessing import LabelEncoder
import joblib

le = LabelEncoder()

df["label"] = le.fit_transform(df["Category"])

num_classes = len(le.classes_)

print("Classes:", le.classes_)
print("Num classes:", num_classes)

# ✅ SAVE LABEL ENCODER (VERY IMPORTANT)
joblib.dump(le, "/content/drive/MyDrive/classification_trans/label_encoder.pkl")

Classes: ['autos' 'entertainment' 'finance' 'foodanddrink' 'health' 'kids'
 'lifestyle' 'movies' 'music' 'news' 'sports' 'travel' 'tv' 'video'
 'weather']
Num classes: 15


['/content/drive/MyDrive/classification_trans/label_encoder.pkl']

## Train-Test Split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

## Dataset Class

In [ ]:
import torch
from torch.utils.data import Dataset

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            padding="max_length",
            truncation=True,
            max_length=128
        )

        item = {k: torch.tensor(v) for k, v in encoding.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))

        return item

## Metrics

In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

## TRAIN FUNCTION (OPTIMIZED)

In [8]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

def train_transformer(model_name):

    print(f"\n🚀 Training {model_name}...\n")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = NewsDataset(X_train.tolist(), y_train.tolist(), tokenizer)
    test_dataset  = NewsDataset(X_test.tolist(), y_test.tolist(), tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes,
        id2label={i: label for i, label in enumerate(le.classes_)},   # ✅ FIX
        label2id={label: i for i, label in enumerate(le.classes_)}    # ✅ FIX
    )

    training_args = TrainingArguments(
        output_dir=f"./results_{model_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=1,
        weight_decay=0.01,
        load_best_model_at_end=True,
        logging_steps=50,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    result = trainer.evaluate()

    return result, model, tokenizer

## Train ONLY BEST 2 MODELS

In [9]:
transformer_models = {
    "BERT": "bert-base-uncased",
    "DistilBERT": "distilbert-base-uncased"
}

results = {}
saved_models = {}

for name, model_name in transformer_models.items():
    result, model, tokenizer = train_transformer(model_name)

    results[name] = result
    saved_models[name] = (model, tokenizer)


🚀 Training bert-base-uncased...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.538135,0.769137,0.776478,0.774103,0.776891,0.776478


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.551309,0.783808,0.772777,0.770046,0.773190,0.772777


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


In [10]:
for name, (model, tokenizer) in saved_models.items():

    path = f"/content/drive/MyDrive/classification_trans/{name}_model"

    model.save_pretrained(path)
    tokenizer.save_pretrained(path)

    print(f"✅ {name} saved at {path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ BERT saved at /content/drive/MyDrive/classification_trans/BERT_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ DistilBERT saved at /content/drive/MyDrive/classification_trans/DistilBERT_model


## Final Comparison

## Save Results to Drive

In [12]:
import pandas as pd

results_df = pd.DataFrame(results).T

results_df.to_csv("/content/drive/MyDrive/classification_trans/results.csv")

print("✅ Results saved")

✅ Results saved


In [13]:
print(results_df)

            eval_loss  eval_accuracy   eval_f1  eval_precision  eval_recall  \
BERT         0.769137       0.776478  0.774103        0.776891     0.776478   
DistilBERT   0.783808       0.772777  0.770046        0.773190     0.772777   

            eval_runtime  eval_samples_per_second  eval_steps_per_second  \
BERT             97.3651                  102.696                 25.677   
DistilBERT       60.0610                  166.481                 41.624   

            epoch  
BERT          1.0  
DistilBERT    1.0  
